<img src="logo.png" width="100" height="100" align="right"> 

## AutoMoL

Pipeline for automated machine learning for drug design.

* **Authors**: Mazen Ahmad, Joris Tavernier, Natalia Dyubankova, Marvin Steijaert
* **Contact**: joris.tavernier@openanalytics.eu, Marvin.Steijaert@openanalytics.eu


&copy; All rights reserved, Open Analytics NV, 2021-2025.

### General Usage Guidelines

We assume that the data provided to the pipeline is clean data. The pipeline **does not handle qualifiers or outliers** in the property target. This is the responsibility of the end-user. This should be done with care and with knowledge of the data at hand. 

### Ignoring all warnings

mainly disables convergence warnings, remove if you would like to get python warnings

In [ ]:
import warnings
import sys, os 
os.environ["BIOSIGNATURE_DEBUG_LOGGING"] = "0"

if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore" # Also affects subprocesses

## Stacking/Blending Classifier
#### Generating Bottleneck features and combining different shallow estimators.
In this notebook we will show how to perform classification for a data set from Therapeutics Data Commons (https://tdcommons.ai/overview). The notebook works with a Dataframe and can easily adopted by reading from csv files. 

In this expert notebook, you can define the method hierarchy or configuration, select used features and choose the methods involved. Additionally to the intermediate notebooks, you can add your own method, change parameter options and add your own feature generation. The AutoMoL concept is detailed below.

<img src="hierarchy.png" width="800" height="800">

Each cell starts with the parameters to be adjusted to your specific case and these parameters are surrounded by *#####################################################*. In the expert notebook, additional variables are set outside this block. This is followed by import statement and the code will be executed in that cell. Each cell is accompanied with a markdown cell detailing the goal of the cell.

### Reading the data 

The dataset is downloaded using the functionality from PyTDC and returns two pandas dataframes (train & test), the variable **name** defines which dataset, the column in the file containing the smiles is set in the variable **smiles_column**. The smiles are then standardized and placed in the column provided in the variable **standard_smiles_column**. If you want to add rdkit descriptors to the feature set, set **check_rdkit_desc** which excludes smiles with nan descriptors.

The **verbose** variable can be set to *1* when more output is desired. 

In [ ]:
#####################################################
#name of the dataset
#name='Caco2_Wang'
#name='HydrationFreeEnergy_FreeSolv'
#name='HydrationFreeEnergy_FreeSolv'
name='PPBR_AZ'
#set the column in the data file where the smiles can be found
smiles_column='Drug'
#verbosity
verbose=1
# set to true if you want to add rdkit descriptors to the feature set
check_rdkit_desc=False
#name of the new column to add the standardized smiles in the dataframe (choice is free)
standard_smiles_column='stand_SMILES'
#####################################################
from tdc.single_pred import ADME
import numpy as np, pandas as pd
from  matplotlib import pyplot as plt
from automol.property_prep import add_stereo_smiles,validate_rdkit_smiles, add_rdkit_standardized_smiles
#dictionary to keep track of the relevant training parameters

data = ADME(name = name)
split = data.get_split()
df=split['train']
test_df=split['test']

model_param={}
model_param['Data file']=name

#in case of reading data from csv
#df= pd.read_csv(file_name, na_values = ['NAN', '?','NaN'])

add_rdkit_standardized_smiles(df, smiles_column,verbose=verbose,outname=standard_smiles_column)
add_rdkit_standardized_smiles(test_df, smiles_column,verbose=verbose,outname=standard_smiles_column)

model_param['Standardization']='rdkit'

if check_rdkit_desc: 
    validate_rdkit_smiles(df, standard_smiles_column,verbose=verbose)

df.dropna(inplace=True, subset = [standard_smiles_column])
df.head(5)

### Building the classes in the case of continuous data
The variable **properties** contains a list of strings. This list corresponds to columns in the data file. These columns are the properties required to be fitted. 

The variable **nb_classes** contains for each property the number of classes and the variable **class_values** contains for each property the splits between classes. In this case the first property is split in classes: <=1 and >1. The second property is split in three classes: <=1; >1 & <=5; and >5.  

If the actual values are unknown, it is possible to set the variables **class_quantiles** as the a list of quantiles for each class. 

The last variable is **min_allowed_class_samples** is the minimum number of samples allowed in a class. If this condition is not met, classes will be based on generated quantiles.

In the case of categorical properties, set **categorical** to True.

In [ ]:
#####################################################
#set the properties to be fitted
categorical=False
properties=['Y']
#total nb of classes for each property
nb_classes=[2]
#set class values for classification
#provide values for splits between classes, e.g. 3 classes, provide 2 values 
#list of values to differentiate between properties
class_values= [[75]]
#set for the use of relative volume values (quantiles)
class_quantiles=None #[ [0.2, 0.4] for p in properties]
min_allowed_class_samples=30
#####################################################

df.dropna(inplace=True,how='all', subset = properties)
df.reset_index(drop=True,inplace=True)

model_param['Properties']=properties
model_param['Categorical properties?']=categorical

if not categorical:
    if class_values is not None:
        model_param['Provided class cutoffs']=class_values
    elif class_quantiles is not None:
        model_param['Provided class quantile cutoffs']=class_quantiles

from automol.property_prep import ClassBuilder

if class_values is not None and class_quantiles is not None:
    print('Both class_values and class_quantiles are set, using values')
    class_quantiles=None
    
if class_quantiles is not None:
    class_values=class_quantiles
    
prop_builder=ClassBuilder(properties=properties,nb_classes=nb_classes,class_values=class_values,
                         categorical=categorical,use_quantiles=class_quantiles is not None,
                         prefix='Class',min_allowed_class_samples=30,verbose=verbose)

#check properties
prop_builder.check_properties(df)        
#prepare properties: e.g. create classes, apply transformations etc..c 
df,train_properties=prop_builder.generate_train_properties(df)

test_df,_=prop_builder.generate_train_properties(test_df)

#retrieve labelnames 
labelnames=prop_builder.labelnames
df_smiles=df[standard_smiles_column]

df.head()

### Sample Weights
set **use_sample_weight** to True if you would like assign more weight to a certain class. The class for each property is set in the dictionary **weighted_samples_index**. 
* In the case of categorical data, use the label. 
* If continuous data is fitted, use the class index, see labelnames (the index for class with the smallest values is 0). 

The weight for each property can be set in **select_sample_weights**. The weights are set in the dataframe with the prefix *sw*. Note that it is possible to directly set the column *sw_\<prop\>* with prop the property name in the dataframe as sample weights for property *prop*. 

In [ ]:
#####################################################
#set to true when planning to use sample_weights, some methods do not have this option
use_sample_weight=False
#set class index for non-categorical data, in this case always the class with the smallest values
weighted_samples_index={p:0 for i,p in enumerate(train_properties)}
#set label for categorical data
if categorical:
    weighted_samples_index={p:'0.0' for p in train_properties}
#set weight of the class specified in property_wclass, other classes have weight 1.0
select_sample_weights={p:10 for p in train_properties}
#####################################################

if use_sample_weight: model_param['Provided samples to weighted per property']=weighted_samples_index
if use_sample_weight: model_param['Provided weights per property']=select_sample_weights

if use_sample_weight:
    df=prop_builder.generate_sample_weights(df,weighted_samples_index,select_sample_weights)
df.head()

### Feature generator(s)
The package provides three feature generators: Bottleneck, rdkit and ECFP. 

* *Bottleneck*: The Bottleneck botlleneck features encoding chemical structure. 
* *rdkit*: RDKit descriptors
* *fps_\<nbits\>_\<radius\>*: Morgan fingerprints from rdkit with provided number of bits and radius. 

See expert notebooks to see how you can add your own features to the AutoMoL framework. 

In [ ]:
########################################################
radius=2
#nbits of ecfp
nbits=2048
##################################################
from automol.feature_generators import retrieve_default_offline_generators
feature_generators=retrieve_default_offline_generators(radius=radius,nbits=nbits)
print(f'available feature generators: {feature_generators.keys()}')

### Self-Made feature generator

In [ ]:
#The code for the self made feature generator is inside the file adme_features.py located in the working directory. 
#for deployment, make sure to add this file to the folder of the model
from adme_features import AdmeTransformer
batch_size=100
seq_len=220    
feature_generators['Adme']=AdmeTransformer(model='/mnt/tree_visualization/all_adme_plus_April.pt' ,use_gpu=False,batch_size=batch_size,seq_len=seq_len,convert_log10=True,feature_properties=None)
    
print('Available feature generator keys')
print(feature_generators.keys())

In [ ]:
try:
    from automol.feature_generators import MolfeatFPVecTransformer
    #works: desc2D, atompair-count, topological-count, fcfp-count, ecfp-count, estate, erg, secfp, pattern, rdkit, fcfp, ecfp, avalon, maccs
    feature_generators['maccs']= MolfeatFPVecTransformer(kind='maccs', dtype=float)
except:
    print('molfeat likely not installed')

### Self-made Classifier

See the python file selfmade_sklearnclassifier.py for the implementation.

In [ ]:
#The code for the self made regressor is inside the file selfmade_sklearnclassifier.py in the working directory. 
#this was required for multiprocessing called from notebook (the class is required to be imported from module)
#for deployment, make sure to add this file to the folder of the model
from selfmade_sklearnclassifier import MLPClassifierWrapper as my_own_mlpclassifier
    
mlp_param_dict={'hidden_layers':[1,2,3,4],
                'hidden_layers_size':[5,10,30,50],
                'learning_rate_init':[1e-4,1e-3,1e-2],
                'max_iter':[100,200,300]}

### Creating the method archives and parameter defaults
The function *get_model_methods_and_parameters* is an interface function that retrieves the model, used method and the parameters to be optimized using default values.

The **task** variable in the case of classification is *Classification* or *clf*. Note that regression is the default.

The default setup only uses the Bottleneck features and no dimension reduction is required. See the expert notebooks for more general use. Not that including fingerprints and rdkit descriptors generally does not lead to increased performance,. 

The variable **scorer** is the metric used when optimizing the parameters.

For more freedom in choosing the methods and hierarchy of base and final estimators, the following variables are required.

the variable **model_config** defines the underlying hierarchy:
* *single_method*: No hierarchy is used, just one method is fitted using the outer folds. 
* *inner_methods*: No final estimator or blender is fitted. A list of classifiers is fitted using the innerfolds and given to a Voting Classifier. The best parameters are chosen per methods for each fold. This means that each fold can provide a new parameter configuration for each method. If the best parameter configuration is already chosen in earlier folds, the model is not added. 
* *inner_stacking*: A sklearn StackingClassifier is directly trained for each outer fold using the provided base and final estimators. These stackingclassifiers are given to a votingclassifier. The number of stackingclassifiers is equal to the number of outer folds. 
* *single_stack*: One StackingClassifier is trained directly using the outerfolds. This fits the base estimators and final estimator in one single cross-validation. The number of base estimators is equal to the given number and the best parameters are chosen in one try.
* *top_method*: Uses the methods found in inner_methods as the list of base estimators. A final estimator is selected from the given final estimators using the outer folds. The output of the base estimators is used as input for the final estimator. 
* *top_stacking*: Same as top_method, but instead of a method as final estimator, a stackingClassifier is fitted. The base estimators found are optimized independently from the final estimator in the stackingClassifier, the final estimator is fitted using the outer folds in a second step. 
* *stacking_stacking*: Uses the stackingmodels found with inner_stacking but instead of a votingclassifier, a stackingclassifier is fitted on top during the outer cross-validation. This is slow!

Selecting the parameter optimization routine can be done by setting several variables. The variable **distribution_defaults** use distribution instead of discrete values for the parameters (where possible), **hyperopt_defaults** switches to HyperOpt optimization. Set **force_gridsearch** or **randomized_iterations** to None if gridsearch is required. Set **randomized_iterations** to a positive integer for randomizedSearch or HyperOpt, recommended for ensemble methods or full stackingClassifiers (inner_stacking and single_stack). 


Selecting the parameter optimization routine can be done by setting several variables. The variable **distribution_defaults** use distribution instead of discrete values for the parameters (where possible), **hyperopt_defaults** switches to HyperOpt optimization. Set **force_gridsearch** or **randomized_iterations** to None if gridsearch is required. Set **randomized_iterations** to a positive integer for randomizedSearch or HyperOpt, recommended for ensemble methods or full stackingregressors (inner_stacking and single_stack). 

Using these options a **method_archive** and a **dim_archive** are created which hold the methods and parameters for respectively regression methods and dimensionality reduction methods. These archives can be further manipulated by changing the defaults parameters for certain methods, adding methods, ... .
* adding a method: the class function is defined as *add_method(self,name,method,param_dictionary)*
    * name: dictionary key used to retrieve the method ('mlp')
    * method: the actual method (MLPClassifierWrapper())
    * param_dictionary: the dictionary of the parameter options of the method (mlp_param_dict)
* adding/changing the value of a parameter options: class function is defined as *add_param(self,name,param_name,param_val)*
    * name: dictionary key ('mlp')
    * param_name: parameter name ('hidden_layers')
    * param_val: list of possible values ([1,2,3,4])

In [ ]:
#####################################################
#Classification or Regression
task='Classification'
#The metric that is optimized when searching the method parameters, examples:
#set scorer : accuracy, balanced_accuracy, f1_weighted, precision, neg_log_loss, roc_auc_ovr_weighted, recall
scorer='balanced_accuracy'

#random state for internal randomness, e.g. randomizedsearch, hyperopt, ...
random_state=5
#random state values for methods, available as a parameter 
random_state_list=[1,7,42,55,3]
#number of jobs used when searching the optimal method parameters (-1 for all)
n_jobs=10
#####################################################
from automol.stacking_util import add_lgbm_xtimes_hyperopt, add_xgb_xtimes_hyperopt
from automol.stacking_methodarchive import ClassifierArchive,ReducedimArchive

model_param['Task']=task
model_param['Score function']=scorer
model_param['Random state']=random_state
model_param['Random state list']=random_state_list

#inner_methods, inner_stacking, single_stack, top_method, top_stacking, single_method
model_config='single_method'

if model_config=='inner_stacking' or model_config=='single_stack' or model_config=='top_stacking':
    use_sample_weight=False


#set to true if gridsearch has to be applied (warning: normally when using inner_stacking or single_stack, gridsearch fails due to the ridiculous amount of parameters)
force_gridsearch=False
#set the value of randomized iterations
randomized_iterations=30
#use distributions for parameter selection, 
#if True a new model is added for method family in each inner fold (the probability that all method parameters are equal in two searches is close to zero)
#also usefull if the number of parameters is too large when using single_stack
distribution_defaults=False 
#use HyperOpt optimization to find the best parameters
hyperopt_defaults=False

#number of jobs used when searching the optimal method parameters (-1 for all)
n_jobs_dict={'outer_jobs':1, #number of jobs/processes for outer cross-validation. if set to None, the value of inner_jobs is evaluated and the available processes are divided among outer and inner folds
            'inner_jobs':-1, #number of jobs/processes for inner cross-validation. -1 means all
            'method_jobs':2} #number of jobs/processes available for the methods. Note that sgd and mlp use blas/mkl and are not affected by this value. (try setting environment variables)
#number of xgb/lgbm threads, None use method_jobs from n_jobs_dict
xgb_threads=None
#number of random forest threads, None use method_jobs from n_jobs_dict
rfr_threads=None
# threads for hyoperopt, set to higher value than 1 to use pyspark, faster for heavy methods, but slower fast methods
hyperopt_threads=None

#naming prefixes for different steps in the pipeline(s)
prefixes={'method_prefix':'clf',
          'dim_prefix':'reduce_dim',
          'estimator_prefix':'est_pipe'}

#create own method archives
method_archive=ClassifierArchive(method_prefix=prefixes['method_prefix'],distribution_defaults=distribution_defaults,hyperopt_defaults=hyperopt_defaults,
                                 random_state=random_state,xgb_threads=xgb_threads,rfr_threads=rfr_threads, method_jobs=n_jobs_dict['method_jobs'])
dim_archive=ReducedimArchive(method_prefix=prefixes['dim_prefix'],distribution_defaults=distribution_defaults,hyperopt_defaults=hyperopt_defaults,
                             random_state=random_state,clf=True)

#adjust parameters options of methods, add methods, etc ...
#method_archive.add_param('rfc','n_estimators',[50,100,200,500])
#method_archive.add_method('knn',KNeighborsClassifier(),{'n_neighbors':[2,3,5,10]})

#add method
mlp_param_dict['random_state']=random_state_list
method_archive.add_method('my_mlp',my_own_mlpclassifier(),mlp_param_dict)

#adding duplicate methods with different random state
nb_copies=8
method='lgbm'
#in the case of hyperopt, it is limited to lgbm and xgb, due to uniqueness requirement in hyperopt parameters. For now workaround is provided for lgbm and xgboost. 
if hyperopt_defaults:
    if method=='lgbm':
        method_archive,clf_list=add_lgbm_xtimes_hyperopt(method_archive,prefixes['method_prefix'],nb_copies,xgb_threads=xgb_threads, distribution_defaults=distribution_defaults,random_state=random_state)
    elif method=='xgb':
        method_archive,clf_list=add_xgb_xtimes_hyperopt(method_archive,prefixes['method_prefix'],nb_copies,xgb_threads=xgb_threads,distribution_defaults=distribution_defaults,random_state=random_state)
else:
    clf_list=method_archive.duplicate_method_xtimes(method_name=method,x=nb_copies,random_state=random_state)
    
#use method archive as blender_archive
blender_archive=method_archive

# print available method keys
print('*****************************************\nkey: Estimator\n*****************************************')
for key,method in method_archive.get_all_method_keys_plus_estimators():
    print(f'\33[1m{key}\33[0m: {method}')
print('*****************************************\nKey: Dimensionality reduction method\n*****************************************')
for key,method in dim_archive.get_all_method_keys_plus_estimators():
    print(f'\33[1m{key}\33[0m: {method}')

### Choosing all methods and used features

Define the methods to select the final estimator or blender from in the variable **blender_list**. Use the estimator keys given above. 

Define the methods for the base estimators in the variable **clf_list**. Use the estimator keys given above. 

Define the used features per property in **used_features**. You can select from *Bottleneck*, *rdkit* or *fps_\<nbits\>_\<radius\>*.  For fingerprints define it as *fps_\<nbits\>_\<radius\>* even if it is not yet present in the given keys (only for fps!)

For dimensionality reduction or feature selection methods, it is a bit more complex. First the default list is set in **red_dim_list**. This list of dimensionality reduction method keys is used if **red_dim_list_per_prop** is None or featurewise dimensionality reduction is not desired but **red_dim_list_per_prop** is defined as featurewise dimensionality reduction (3 nested loops). 

The variable **local_dim_red** can be set to use dimensionality reduction methods for each type of features individually. If not set, the different types of features are given as one to a dimensionality reduction method. If **local_dim_red** is set, each feature type is given to its own dimensionality reduction method. 

The variable **red_dim_list_per_prop** is a nested list of dimensionality reduction method keys. Two options are available a list of list or a list of lists of lists, e.g. double nested or triple nested. The outer loop corresponds in both cases to the property. For each property you have to define the dimensionality reduction keys. Next, the result depends on **local_dim_red**.
* **local_dim_red**=True:
    * provide a single list of keys: all features for that property will select their individual dimensionality reduction method from this list
    * provide a list of lists of keys: here you provide for each feature type a list of dimensionality reduction keys. This means each feature type will select from his individual list. 
* **local_dim_red**=False:
    * provide a single list of keys: the dimensionaltiy reduction methods for that property will be selected from this list
    * provide a list of list of keys: **mismatch** since all features are given to the same dimensionality reduction method. The defaults methods in **red_dim_list** are used. 
 
The class *ModelAndParams* will now be used to set everything in the right place and retrieve the model, parameters options and parameter search object. 

In [ ]:
#define used methods
blender_list=['lr','my_mlp']#+[clf_list[0]]#,'rfc']
#define base estimators or used method for single_method
clf_list=['lr' ,'SVC']#,'my_mlp']#+clf_list[:3]#, 'lda','rfc']#,'qda','sgdc','dtc','xgb']

#select Features for each property, for fps define it as fps_<nbits>_<radius> even if it is not yet present in the given keys (only for fps!)
used_features=[ ['Bottleneck','maccs']]

#define default dimensionality reduction methods, can be overwritten by red_dim_list_per_prop
#these are used in the case that red_dim_list_per_prop is None or 
#if local_dim_red is not set but feature wise dimensionality reduction methods are provided (3 nested loops)
#if pca does not have enough memory, the kernel randomly stops
red_dim_list=['passthrough','v_threshold']#['passthrough','pca','SelectPercentile','v_threshold','Kbest','rfe','frommodel']  
#select dimensionality reduction methods for each property, 
#this can be feature type specific per property or try the same for all feature types, 
#thus this can be 3 nested lists (property, feature, dim reduction methods) or 2 (property and dim reduction methods)
red_dim_list_per_prop=[
                  ['passthrough'], #Property 1: one list for all feature types, try these for all feature types
                  #['passthrough'] # #Property 2: list of lists feature type specific for this property
                  ]
#set this boolean for featuretype specific dimensionality reduction,
#if False all features are concatenated first and then given to dimensionality reduction, uses red_dim_list if red_dim_list_per_prop[i] is a double nested list for prop i
#when True each feature set has its own dimensionality reduction type
local_dim_red=True

#number of components for dimensionality reduction, important for hyperopt when number of samples is less than number of features and less than required dimension.
#For gridsearch and randomizedsearch, we can easily access the parameters and adjust internally to min(dim,num_samples,num_features)
if hyperopt_defaults:
    dim_red_n_components=[10,25,50,100,200,300,500]
else:
    #use internal AutoML defaults
    dim_red_n_components=None
#####################################################
from automol.stacking_util import ModelAndParams
if model_config=='single_method' and len(clf_list)>1:
    print(f'Warning: more methods are given in clf_list and hierarchy is set to {model_config}, only {clf_list[0]} will be used')

if red_dim_list_per_prop is not None:
    assert len(red_dim_list_per_prop)==len(properties), 'provided list of dim reduction methods per peroprty must equal length of list of properties'
    model_param['red_dim_list']=red_dim_list_per_prop
else:
    model_param['red_dim_list']=red_dim_list
    
if dim_red_n_components is not None: model_param['dim_red_n_components']=dim_red_n_components
model_param['local_dim_red']=local_dim_red
model_param['clf_list']=clf_list
model_param['blender_list']=blender_list
model_param['force_gridsearch']=force_gridsearch
model_param['randomized_iterations']=randomized_iterations
model_param['distribution_defaults']=distribution_defaults
model_param['hyperopt_defaults']=hyperopt_defaults  
model_param['used_features']=used_features

param_Factory=ModelAndParams(task=task,
                             distribution_defaults=distribution_defaults, #use distributions for parameter selection in randomized search
                             hyperopt_defaults=hyperopt_defaults, #use hyperopt
                             use_gpu=False,
                             normalizer=True, #normalize input base estimators
                             top_normalizer=False,#normalize output base estimators and thus input top estimator
                             random_state=random_state_list,#random_state
                             red_dim_list=red_dim_list,#list of dimensionality reducing methods keys from dim_archive
                             method_list=clf_list,#list of classifier keys from method_archive
                             blender_list=blender_list,#list of classifier keys from blender_archive
                             model_config=model_config, #string with the model layout/config
                             randomized_iterations=randomized_iterations, #number of randomized iterations
                             n_jobs_dict=n_jobs_dict,
                             use_sample_weight=use_sample_weight,
                             feature_generators=feature_generators,
                             local_dim_red=local_dim_red,
                             dim_red_n_components=dim_red_n_components,
                             method_archive=method_archive, #archive class holding the methods
                             dim_archive=dim_archive, #archive class holding the dimensionality reducing methods
                             blender_archive=blender_archive, #archive class holding the blender methods
                             prefixes=prefixes #naming convention in parameters for grid/randomized search
                            ,hyperopt_threads=hyperopt_threads, #overwrites  n_jobs_dict['inner_jobs'] in the case of hyperopt_defaults
                             labelnames=labelnames,
                             verbose=verbose)
stacked_model, prefixes,params_grid,blender_params,paramsearch = param_Factory.get_model_and_params()

### Defining your own clustering algorithm

You can define your own clustering algorithm following the interface provided in selfmade_clustering.py. In essence smiles are provided and you set the assigned cluster. The class SklearnClusteringForSmiles takes as input a clustering algorithm that has the functionality fit_predict(X) with X and 2d matrix. This 2d matric is generated using the provided feature generators. 


In [ ]:
from selfmade_clustering import SklearnClusteringForSmiles as my_own_clustering #(also present in automol.clustering)
from sklearn.cluster import SpectralClustering
# see https://scikit-learn.org/stable/modules/generated/sklearn.cluster.SpectralClustering.html
#slow
#sklearn_clustering=SpectralClustering(n_clusters=30, assign_labels='cluster_qr', random_state=0)

#https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html#sklearn.cluster.AgglomerativeClustering
from sklearn.cluster import AgglomerativeClustering
sklearn_clustering=AgglomerativeClustering(n_clusters=30)

#https://scikit-learn.org/stable/modules/generated/sklearn.cluster.BisectingKMeans.html#sklearn.cluster.BisectingKMeans
from sklearn.cluster import BisectingKMeans
#sklearn_clustering = BisectingKMeans(n_clusters=30, random_state=0)


#create your clustering method 
Validation_set_clustering_algo=my_own_clustering(feature_generators=feature_generators,
                                                          used_features=['Bottleneck'],
                                                          random_state=42,
                                                          sklearn_estimator=sklearn_clustering)

from automol.feature_generators import ECFPGenerator
from automol.clustering import HierarchicalButina
#define a clustering algorithm for property cliffs
#Note that butina requires a featuregenerator able to generate the rdkit bitvec arrays. Only ECFPGenerator has this functionality in automol.feature_generators  
property_cliff_clustering= HierarchicalButina(cutoff=[0.3,0.5], feature_generator=ECFPGenerator(radius=3,nBits=2048))


### Splitting the data in Training and Validation sets
The smiles are clustered and split into a training and a validation set. 

It is possible to choose between the following clustering methods by setting the variable **val_clustering** to:
* *Scaffold*: performs MurckoScaffold clustering and uses the boolean **val_include_chirality**
* *Butina*: performs Butina clustering based on the variable **val_butina_cutoff** and afterwards the smiles are assigned to the closest centroid and possibly not the centroid they were assigned to in the first place. 
* *HierarchicalButina*: Does applies butina clustering hierarchically, here the variable **val_butina_cutoff** should contain a list of increasing cutoff values.
* *Bottleneck*: performs Kmeans++ on the generated Bottleneck features. The value of K and thus the number of cluster is fixed by the variable **val_km_groups**

Validation **strategy**:
* *mixed*: 30% activity or property cliffs, 30% leave group out and rest stratified. Values can be altered.
* *leave group out*: full clusters in validation set.
* *stratified*: samples from each cluster are added to the validation set.

The variable **minority_nb** is the threshold to define clusters as small. If a cluster contains less samples than this value, this cluster added to the cluster *minorities*. This only happens for the stratified and mixed validation strategies.

In [ ]:
#####################################################
#mixed, 
strategy='lgo'
# ratio of size Test set/full data
test_size=0.25
# ratio of size validation set/training+validation data
val_size=0.2
# Small clusters are grouped in one larger cluster for stratified validation (and thus also mixed). 
# This value is the threshold to determine which clusters are 'small'
minority_nb=5
#####################################################
from automol.validation import stratified_validation, leave_grp_out_validation, mixed_validation

#split the data in test and validation+training
leave_group_out=None
prop_cliff_dict_test=None
if strategy=='mixed':
    rel_prop_cliff=0.5
    #stratified is essentially the remaining number of samples left to reach the desired amount of test samples set by test_size ratio
    #    and is therefore not actually used
    mix_coef_dict={ 'prop_cliffs': 0.3,'leave_group_out': 0.3 ,'stratified': 0.4}
    Train, Validation,leave_group_out, prop_cliff_dict = mixed_validation(df_orig=df,properties=properties,stacked_model=stacked_model,standard_smiles_column=standard_smiles_column,
                                                    prop_cliff_cut=rel_prop_cliff,test_size=test_size,
                                                    verbose=verbose,random_state=random_state,mix_dict=mix_coef_dict,
                                                    categorical_data=categorical,minority_nb=minority_nb,
                                                  clustering_algorithm=Validation_set_clustering_algo,chem_clustering_algorithm=property_cliff_clustering)
elif strategy=='stratified':
    Train, Validation =  stratified_validation(df,train_properties,stacked_model,standard_smiles_column,df_smiles,test_size=test_size,
                                                  verbose=verbose,random_state=random_state,clustering_algorithm=Validation_set_clustering_algo, minority_nb=minority_nb)
else:
    Train, Validation =  leave_grp_out_validation(df,train_properties,stacked_model,standard_smiles_column,df_smiles,test_size=test_size,
                                                  verbose=verbose,clustering_algorithm=Validation_set_clustering_algo, random_state=random_state)
    leave_group_out=np.arange(len(Validation))
                
    

#placing the validation and training set inside the model generator
stacked_model.Validation=Validation
stacked_model.Train=Train
stacked_model.smiles=standard_smiles_column # set col smiles name

### Clustering the data for nested cross-validation
same options as for clustering the data for splitting in training and validation sets

In [ ]:
#property check
cv_clustering='Bottleneck'
km_groups=20
for p in properties:
    prop_count=df[p].count()
    if cv_clustering=='Bottleneck' and prop_count/km_groups<10:
        print('Warning: on average less than 10 samples per cluster')
        
#clustering the data

#https://scikit-learn.org/stable/modules/generated/sklearn.cluster.BisectingKMeans.html#sklearn.cluster.BisectingKMeans
from sklearn.cluster import BisectingKMeans
sklearn_clustering = BisectingKMeans(n_clusters=20, random_state=0)

train_clustering_algo=my_own_clustering(feature_generators=feature_generators,
                                                          used_features=['Bottleneck'],
                                                          random_state=42,
                                                          sklearn_estimator=sklearn_clustering)

stacked_model.Data_clustering(random_state=random_state,clustering_algorithm=train_clustering_algo)

### Searching models for the properties using cross-validation
The different methods are fitted using the chosen parameters above.

The cross-validation is determined by the parameter **outer_folds**, e.g. the number of outer folds (number of inner folds is outer - 1) and the **cross_val_split**. Choose for the latter between *LGO*: leave one group out, *GKF*: group k-fold which splits full groups in cross-validation manner or *SKF*: stratified K-fold which makes sure that the split preserves the ratio of samples between classes (only recommended when the number of samples is really low for one class, try GKF first and if GKF fails try SKF). 

In [ ]:
%%time  
#####################################################
#cross-validation parameters
#select cross-validation split:  LGO, SKF or GKF
cross_val_split='GKF'
outer_folds=4
#####################################################
model_param['Used cross-validation technique']=cross_val_split
if cross_val_split != 'LGO': model_param['Number of folds']=outer_folds


results_dictionary={}
#empty to put this one first (keep in mind order in dictionary is not reliable)
for index,p in enumerate(train_properties):
    results_dictionary[p]={}      
    sample_weight=None
    if use_sample_weight:
        sample_weight=stacked_model.Train[f'sw_{p}'].values
    
    features=used_features[index]
    if 'rdkit' in features and not check_rdkit_desc:
        print('Warning, smiles are not validated for nan rdkit descriptors')
    
    dim_list=None
    if red_dim_list_per_prop is not None:
        dim_list=red_dim_list_per_prop[index]
        if isinstance(dim_list[0],list) and not local_dim_red:
            dim_list=None
        elif isinstance(dim_list[0],list):
            assert len(dim_list)==len(features), 'Number of provided features must equal number of dimensionality reduction method lists in the case of featurewise specific dimensionality reduction / feature selection'       
    
    params_grid,blender_params = param_Factory.get_feature_params(selected_features=features,dim_list=dim_list)
    
    stacked_model.search_model(df= None,   prop=p,  smiles=standard_smiles_column,
                                params_grid=params_grid,
                                paramsearch=paramsearch,
                               features=features,
                              scoring=scorer,#'neg_mean_absolute_error',#
                              cv=outer_folds-1,  outer_cv_fold=outer_folds, split=cross_val_split, 
                              use_memory=True,
                              plot_validation=True, 
                             refit=False,# no refit with validation. comes later,
                             blender_params=blender_params
                              ,prefix_dict=prefixes,random_state=random_state,sample_weight=sample_weight
                              ,results_dict=results_dictionary)
model_str=stacked_model.print_metrics()

### Validation
show the results on the validation set. The variable **cmap** is a list of colormaps used for the figures.

In [ ]:
#####################################################
#set colormap for classification report and confusion matrix: PiYG, viridis, YlGnBu, Blues
cmap=['PiYG','Blues']
#####################################################
from automol.stacking_util import ResultDesigner

out=stacked_model.predict( props =None, smiles=stacked_model.Validation[standard_smiles_column],compute_SD=True,convert_log10=False)

print('##########################################################')
print('                    Validation set')
youden_dict=ResultDesigner().show_classification_report(train_properties,out,[stacked_model.Validation[f'{p}'].values for p in train_properties],labelnames=labelnames,cmap=cmap)
F1_dict=ResultDesigner().show_clf_threshold_report(train_properties,out,[stacked_model.Validation[f'{p}'].values for p in train_properties],labelnames=labelnames,youden_dict=youden_dict)

### Test set

In [ ]:
print('##########################################################')
print('                    Test set')
out_test=stacked_model.predict( props =None, smiles=test_df[standard_smiles_column],compute_SD=True,convert_log10=False)

_=ResultDesigner().show_classification_report(train_properties,out_test,[test_df[f'{p}'].values for p in train_properties],labelnames=labelnames,cmap=cmap)
_=ResultDesigner().show_clf_threshold_report(train_properties,out_test,[test_df[f'{p}'].values for p in train_properties],labelnames=labelnames,youden_dict=youden_dict)

### Generate plotly figures

The variable **show_plotly_figures** can be set to False if you experience 403 errors when trying to save the notebook (For certain browsers this can happen, clear the output from this cell and the notebook wil save).

The variables **show_adv** and **show_th** can be set to display advanced classifiers figures and threshold tuning figures respectively

In [ ]:
#import plotly.offline as py
######################################################
show_plotly_figures=True
show_adv=True
show_th=True
x_width=600
y_height=450
#######################################################
from automol.plotly_util import PlotlyDesigner, create_figure_captions, get_figures_from_types
youden_dict,fig_l=PlotlyDesigner().show_classification_report(train_properties,out_test,[test_df[f'{p}'].values for p in train_properties],
                                                              labelnames=labelnames,cmap=cmap,fig_size=(x_width,y_height)
                                              ,results_dict=results_dictionary)
fig_adv=PlotlyDesigner().show_additional_classification_report(train_properties,out_test,[test_df[f'{p}'].values for p in train_properties],
                                                               labelnames=labelnames,cmap=cmap,fig_size=(x_width,y_height)
                                              ,results_dict=results_dictionary)
F1_dict,fig_th=PlotlyDesigner().show_clf_threshold_report(train_properties,out_test,[test_df[f'{p}'].values for p in train_properties],youden_dict=youden_dict,
                                                          labelnames=labelnames,fig_size=(x_width,y_height)
                                              ,results_dict=results_dictionary)

plotly_dictionary={**fig_l,**fig_th,**fig_adv}
plotly_show={**fig_l}
if show_th:    
    plotly_show={**plotly_show,**fig_th}
if show_adv:
    plotly_show={**plotly_show,**fig_adv}

_,_,type_keys=create_figure_captions(plotly_dictionary.keys())
#retrieve figure keys from types ordered per property
sorted_keys=get_figures_from_types(plotly_dictionary=plotly_show,properties=train_properties,types=type_keys)
#retrieve captions in order of sorted keys
captions,types,_=create_figure_captions(sorted_keys)

if show_plotly_figures:
    for index,key in enumerate(sorted_keys):
        plotly_dictionary[key].show()
        print(captions[index])



### Generate pdf summary
Generate a small report for the model. You can set the authors by defining **authors**, the title by **title**, contact information in **mails** and add a small summary in **summary**. You can choose the types of figures to be added to the report in **figures_types** as well as the number of columns for these figures in **nb_columns**. The name of the pdf file can be set in **output_file**.  

In [ ]:
print('Figure types:\n')
for it,t in enumerate(types):
    print(f"{it+1}. "+t+"\n")

In [ ]:
#####################################################
output_file="summary.pdf"
title='AutoMoL report Title'
authors='Joris Tavernier'
mails='joris.tavernier@openanalytics.eu'
summary="""Project details -- Heri, inquam, ludis commissis ex urbe profectus veni ad vesperum.
causa autem fuit huc veniendi ut quosdam hinc libros promerem. et
quidem, Cato, hanc totam copiam iam Lucullo nostro notam esse oportebit;
nam his libris eum malo quam reliquo ornatu villae delectari. est enim
mihi magnae curae - quamquam hoc quidem proprium tuum munus est - ut ita
erudiatur, ut et patri et Caepioni nostro et tibi tam propinquo
respondeat. laboro autem non sine causa; nam et avi eius memoria moveor
- nec enim ignoras, quanti fecerim Caepionem, qui, ut opinio mea fert,
in principibus iam esset, si viveret - et Lucullus mihi versatur ante
oculos, vir cum virtutibus omnibus excellens, tum mecum et amicitia et
omni voluntate sententiaque coniunctus."""
#use self provided template from notebook directory
#template_file="summary.html"
#Set template_file to None to use package template
template_file=None
#select figure types to be added to the report
figure_types=['A','B','H','I']
nb_columns=5
#####################################################
from automol.plotly_util import generate_report

#retrieve figure keys from types ordered per property
selected_figures=get_figures_from_types(plotly_dictionary=plotly_dictionary,properties=train_properties,types=figure_types)

captions,types,_=create_figure_captions(selected_figures)
pdf_data=generate_report(plotly_dictionary,selected_figures,captions,types,title=title, authors=authors,
                         mails=mails,summary=summary,model_param=model_param,model_tostr_list=model_str,properties=train_properties,template_file=template_file,nb_columns=nb_columns )
with open(output_file, mode="wb") as f:
    f.write(pdf_data)

### Saving the model(s)
the models wil be saved in **save_model_to_file**. The model is refitted on both the training and validation set. The model(s) will additionally be automatically and internally evaluated on a few public smiles for reproducability, which can be verified when loading the model.  

In [ ]:
from automol.plotly_util import save_result_dictionary_to_json
#####################################################
#the filename where the model will be saved
json_file='results_dictionary.json' 
yaml_file='training_param.yaml'
#####################################################
save_result_dictionary_to_json(result_dict=results_dictionary,json_file=json_file)
#import yaml
#with open(yaml_file, 'w') as file:
#    documents = yaml.dump(model_param, file)


In [ ]:
#####################################################
#the filename where the model will be saved
save_model_to_file=f'{name}_stackingclfmodel.pt'
#####################################################
from automol.stacking import save_model
stacked_model.clean()
#placing the test, validation and training set inside the model generator
for p in train_properties:
    sample_train=None
    sample_val=None
    if use_sample_weight:
        sample_train=stacked_model.Train[f'sw_{p}'].values
        sample_val=stacked_model.Validation[f'sw_{p}'].values
    stacked_model.refit_model(models=p,sample_train=sample_train,sample_val=sample_val,prefix_dict=prefixes)
## clean the class first by removing the computed features
stacked_model.clean()
stacked_model.compute_SD=True
save_model(stacked_model, save_model_to_file)
stacked_model.validate(df=None, # df with smiles and the properties
                       props=None, # name of the task 
                    true_props=None,# name of the property in df
                    smiles=standard_smiles_column)


### Loading the model
The reproducable output is automatically verified and the maximum error for each property is printed if the check fails. If not a succes message is printed.  

In [ ]:
from automol.stacking import load_model

stacked_model2= load_model( save_model_to_file,use_gpu=False) 
stacked_model2.print_metrics()

### Saving the notebook and the output as html

Save the notebook first and then execute the cell. This will save this notebook, set the name in variable **notebook_name** without the .ipynb extension. The html will then be saved in **save_output_to_file** 

In [ ]:
#####################################################
#the name of this notebook, is important for saving the output (without the .ipynb)
notebook_name='Expert_Classifier'
#html file where the output is
save_output_to_file='Cypro_expertclfmodel.html'
#####################################################
notebook_file=notebook_name+'.ipynb'
notebook_html=notebook_name+'.html'
!jupyter-nbconvert --to html $notebook_file
!mv $notebook_html $save_output_to_file